# MEIKURAL Version 3 — native STFT inference

Raw audio → mono/resampling → 30-second windows → RMS/gain normalization → zero padding → native STFT magnitude → log intensity compression → larger CNN.

There is **no tonic estimation, frequency alignment, frequency interpolation, or image resizing**. The default input is the full **4,097 × 1,292** matrix. The CNN uses 32/64/128 filters and a 256-unit hidden classifier. This notebook uses only Version 3 checkpoints; train your model first.

Use the Python (MEIKURAL) kernel and run cells in order. Whole-song and window scores are comparative scores over seven raags, not calibrated probabilities.

The default checkpoint is `models/ver3/latest.pt`; preparation and training must be run separately before inference.

**Shared test audio:** all versions use `data/test_ver1_handpicked`, including its `marwa` folder. Checkpoints, feature caches, and reports remain version-specific. After adding files, rerun the recording-list and batch cells (or Run All) before reading the confusion matrices. Saved outputs are not refreshed automatically.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src/ver3/config.json").is_file(), "Launch from the repository root or notebooks/"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.ver3.config import load_config, resolve_path
from src.ver3.inference.predict import predict

config = load_config()


## Choose the audio and checkpoint

Edit these two paths. Relative audio paths are relative to the repository. The checkpoint supplies its own saved preprocessing and model settings; current config edits cannot alter the trained representation.

In [ ]:
TEST_DIR = ROOT / "data/test_ver1_handpicked"
AUDIO_PATH = TEST_DIR / 'jog/Venkatesh_kumar.mp3'
CHECKPOINT_PATH = resolve_path(config["paths"]["run"]) / "latest.pt"


In [ ]:
result = predict(AUDIO_PATH, CHECKPOINT_PATH)
print(result["message"])
print(f"Duration: {result['duration_seconds']:.3f} s | Device: {result['device']}")
print(f"Usable windows: {result['usable_windows']} | Failed windows: {result['failed_windows']}")


In [ ]:
import pandas as pd
from IPython.display import display

print("Overall ranking (real-duration-weighted window scores)")
display(pd.DataFrame(result["ranking"]))
columns = ["start_seconds", "valid_start_seconds", "valid_seconds", "left_padding_seconds",
           "right_padding_seconds", "top_raag", "failure"]
display(pd.DataFrame(result["windows"])[columns])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

successful = [row for row in result["windows"] if row["scores"] is not None]
if successful:
    times = [row["valid_start_seconds"] + row["valid_seconds"] / 2 for row in successful]
    scores = np.asarray([row["scores"] for row in successful])
    fig, ax = plt.subplots(figsize=(12, 4))
    for index, label in enumerate(result["classes"]):
        ax.plot(times, scores[:, index], marker=".", label=label)
    ax.set(xlabel="Window midpoint in recording (seconds)", ylabel="Comparative score",
           ylim=(0, 1), title="MEIKURAL — window predictions")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
    fig.tight_layout()
    plt.show()
else:
    print("No score timeline: inspect window failures above.")


## Interpretation

The overall ranking uses the same real-duration-weighted mean across successful windows as earlier versions. Padded windows contribute only their real duration. Overlapping windows are correlated.

Pitch and singer tonic differences remain visible at different absolute frequencies. The model must learn from that variation; no tonic is estimated or corrected. Failed/silent windows are reported. This untrained Version 3 implementation makes no accuracy claim.